# 🔍 Heading & Meta Audit
Audit SEO on-page secara bulk untuk mendeteksi issue pada heading dan meta tag.

---
| Cek | Keterangan |
|---|---|
| **Duplicate H1** | Konten H1 yang sama muncul di lebih dari satu halaman |
| **Multiple H1** | Satu halaman memiliki lebih dari satu tag H1 |
| **Missing H1** | Halaman tidak memiliki tag H1 sama sekali |
| **Hierarchy violation** | Heading loncat level (misal H2 → H4), tidak diawali H1, atau H1 bukan heading pertama |
| **Missing title** | Tag `<title>` tidak ada atau kosong |
| **Missing meta desc** | Tag `<meta name="description">` tidak ada atau kosong |

**Output:** 2 sheet CSV — summary per URL & detail per issue

## 📦 Install Dependencies

In [ ]:
!pip install requests beautifulsoup4 pandas openpyxl -q

## ⚙️ Konfigurasi & Upload
Upload file `.txt` berisi daftar URL yang akan diaudit (satu URL per baris).

In [ ]:
from google.colab import files

print("📂 Silakan upload file .txt berisi daftar URL...")
uploaded = files.upload()

if not uploaded:
    raise ValueError("❌ Tidak ada file yang diupload!")

INPUT_FILE       = list(uploaded.keys())[0]
OUTPUT_SUMMARY   = "audit_summary.csv"    # Satu baris per URL
OUTPUT_DETAIL    = "audit_detail.csv"     # Satu baris per issue

print(f"\n✅ File diterima     : {INPUT_FILE}")
print(f"💾 Output summary    : {OUTPUT_SUMMARY}")
print(f"💾 Output detail     : {OUTPUT_DETAIL}")

## 🚀 Jalankan Audit

In [ ]:
# ============================================================
# Heading & Meta Audit
# Checks  : duplicate h1, multiple h1, missing h1,
#            heading hierarchy violations,
#            missing title, missing meta description
# Output  : summary CSV (per URL) + detail CSV (per issue)
# Saves   : real-time ke CSV (aman jika berhenti di tengah)
# ============================================================

import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urlsplit, urlunsplit, quote

# ── Config ───────────────────────────────────────────────────
DELAY_SECONDS   = 5
REQUEST_TIMEOUT = 30

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.8,id;q=0.7",
    "Connection": "keep-alive",
})

# ── Logger ───────────────────────────────────────────────────
def log(level: str, msg: str):
    icons = {
        "INFO":  "ℹ️ ",
        "OK":    "✅",
        "ISSUE": "🚨",
        "WARN":  "⚠️ ",
        "ERROR": "❌",
        "SAVE":  "💾",
        "DONE":  "🎉",
        "DELAY": "⏳",
    }
    ts   = datetime.now().strftime("%H:%M:%S")
    icon = icons.get(level, "  ")
    print(f"[{ts}] {icon} [{level}] {msg}")

# ── URL Encoder ──────────────────────────────────────────────
def encode_url(url: str) -> str:
    """Encode non-ASCII characters in URL path (e.g. CJK slugs)."""
    try:
        parts = urlsplit(url)
        return urlunsplit(parts._replace(
            path=quote(parts.path, safe="/"),
            query=quote(parts.query, safe="=&"),
        ))
    except Exception:
        return url

# ── Heading Hierarchy Checker ─────────────────────────────────
def check_heading_hierarchy(headings: list[tuple[int, str]]) -> list[str]:
    """
    Terima list of (level, text) dan kembalikan list string issue.

    Deteksi:
      1. H1 bukan heading pertama di halaman
      2. Tidak diawali H1 (heading pertama bukan H1)
      3. Loncat level (non-sequential) misal H2 → H4
    """
    issues = []
    if not headings:
        return issues

    levels = [h[0] for h in headings]

    # Cek 1: heading pertama bukan H1
    if levels[0] != 1:
        issues.append(f"First heading is H{levels[0]}, not H1")

    # Cek 2: ada H1 tapi bukan yang pertama
    h1_positions = [i for i, l in enumerate(levels) if l == 1]
    if h1_positions and h1_positions[0] != 0:
        issues.append(f"H1 found at position {h1_positions[0] + 1}, not at the top")

    # Cek 3: loncat level (non-sequential)
    for i in range(1, len(levels)):
        prev, curr = levels[i - 1], levels[i]
        # Boleh naik level (H3→H2) atau turun 1 level (H2→H3)
        # Tidak boleh loncat lebih dari 1 level ke bawah
        if curr > prev + 1:
            issues.append(
                f"Heading jump: H{prev} ('{headings[i-1][1][:40]}') "
                f"→ H{curr} ('{headings[i][1][:40]}')"
            )

    return issues

# ── Page Auditor ─────────────────────────────────────────────
def audit_page(url: str) -> dict:
    """
    Audit satu URL.
    Return:
      summary dict  → satu baris untuk audit_summary.csv
      issues list   → list of dicts untuk audit_detail.csv
    """
    summary = {
        "url":                    url,
        "status":                 None,
        "title":                  None,
        "meta_description":       None,
        "h1_count":               0,
        "h1_texts":               None,
        "has_missing_title":      False,
        "has_missing_meta_desc":  False,
        "has_missing_h1":         False,
        "has_multiple_h1":        False,
        "has_hierarchy_issue":    False,
        "issue_count":            0,
    }
    issues = []

    def add_issue(issue_type: str, detail: str):
        issues.append({
            "url":        url,
            "issue_type": issue_type,
            "detail":     detail,
        })

    # HTTP Request
    try:
        resp = SESSION.get(encode_url(url), timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        summary["status"] = f"{resp.status_code} OK"
    except Exception as e:
        summary["status"] = f"error: {e}"
        return summary, issues

    soup = BeautifulSoup(resp.text, "html.parser")

    # ── Meta: Title ───────────────────────────────────────────
    title_tag = soup.find("title")
    title_text = title_tag.get_text(strip=True) if title_tag else ""
    summary["title"] = title_text or None

    if not title_text:
        summary["has_missing_title"] = True
        add_issue("missing_title", "<title> tag is missing or empty")

    # ── Meta: Description ────────────────────────────────────
    meta_desc_tag = soup.find("meta", attrs={"name": lambda v: v and v.lower() == "description"})
    meta_desc = meta_desc_tag.get("content", "").strip() if meta_desc_tag else ""
    summary["meta_description"] = meta_desc or None

    if not meta_desc:
        summary["has_missing_meta_desc"] = True
        add_issue("missing_meta_description", "<meta name='description'> is missing or empty")

    # ── Headings ─────────────────────────────────────────────
    heading_tags = soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"])
    headings = [(int(tag.name[1]), tag.get_text(strip=True)) for tag in heading_tags]

    h1_texts = [text for level, text in headings if level == 1]
    h1_count = len(h1_texts)
    summary["h1_count"] = h1_count
    summary["h1_texts"] = " | ".join(h1_texts) if h1_texts else None

    # Missing H1
    if h1_count == 0:
        summary["has_missing_h1"] = True
        add_issue("missing_h1", "No H1 tag found on this page")

    # Multiple H1
    if h1_count > 1:
        summary["has_multiple_h1"] = True
        add_issue(
            "multiple_h1",
            f"{h1_count} H1 tags found: {' | '.join(h1_texts)}"
        )

    # Heading Hierarchy
    hierarchy_issues = check_heading_hierarchy(headings)
    if hierarchy_issues:
        summary["has_hierarchy_issue"] = True
        for hi in hierarchy_issues:
            add_issue("heading_hierarchy_violation", hi)

    summary["issue_count"] = len(issues)
    return summary, issues

# ── Duplicate H1 Checker ─────────────────────────────────────
def flag_duplicate_h1(summaries: list[dict]) -> list[dict]:
    """
    Setelah semua URL diproses, tandai H1 yang muncul
    di lebih dari satu halaman sebagai duplicate.
    """
    from collections import Counter
    h1_counter = Counter()
    for s in summaries:
        if s.get("h1_texts"):
            for h1 in s["h1_texts"].split(" | "):
                h1_counter[h1.strip()] += 1

    duplicate_h1s = {h1 for h1, count in h1_counter.items() if count > 1}

    detail_rows = []
    for s in summaries:
        if s.get("h1_texts"):
            for h1 in s["h1_texts"].split(" | "):
                h1 = h1.strip()
                if h1 in duplicate_h1s:
                    s["has_duplicate_h1"] = True
                    detail_rows.append({
                        "url":        s["url"],
                        "issue_type": "duplicate_h1",
                        "detail":     f"H1 '{h1}' appears on {h1_counter[h1]} pages",
                    })
        if "has_duplicate_h1" not in s:
            s["has_duplicate_h1"] = False

    return detail_rows

# ── CSV Helpers ───────────────────────────────────────────────
SUMMARY_COLS = [
    "url", "status", "title", "meta_description",
    "h1_count", "h1_texts",
    "has_missing_title", "has_missing_meta_desc",
    "has_missing_h1", "has_multiple_h1", "has_duplicate_h1",
    "has_hierarchy_issue", "issue_count",
]
DETAIL_COLS = ["url", "issue_type", "detail"]

def init_csvs():
    pd.DataFrame(columns=SUMMARY_COLS).to_csv(OUTPUT_SUMMARY, index=False, encoding="utf-8")
    pd.DataFrame(columns=DETAIL_COLS).to_csv(OUTPUT_DETAIL, index=False, encoding="utf-8")
    log("SAVE", f"CSV diinisiasi: {OUTPUT_SUMMARY} & {OUTPUT_DETAIL}")

def append_summary(row: dict):
    # Pastikan has_duplicate_h1 ada (diisi setelah loop selesai)
    if "has_duplicate_h1" not in row:
        row["has_duplicate_h1"] = False
    pd.DataFrame([row])[SUMMARY_COLS].to_csv(
        OUTPUT_SUMMARY, mode="a", header=False, index=False, encoding="utf-8"
    )

def append_details(rows: list[dict]):
    if rows:
        pd.DataFrame(rows)[DETAIL_COLS].to_csv(
            OUTPUT_DETAIL, mode="a", header=False, index=False, encoding="utf-8"
        )

# ── Entry Point ───────────────────────────────────────────────
def main():
    try:
        with open(INPUT_FILE, "r", encoding="utf-8") as f:
            urls = [ln.strip() for ln in f if ln.strip()]
    except FileNotFoundError:
        log("ERROR", f"File tidak ditemukan: {INPUT_FILE}")
        return

    if not urls:
        log("WARN", "Tidak ada URL di file input.")
        return

    total = len(urls)
    log("INFO", f"Ditemukan {total} URL — mulai audit...")
    print("═" * 60)

    init_csvs()

    all_summaries = []
    all_details   = []
    total_issues  = 0

    for i, url in enumerate(urls, start=1):
        print(f"\n── [{i}/{total}] ──────────────────────────────────")
        log("INFO", f"URL     : {url}")

        summary, issues = audit_page(url)
        all_summaries.append(summary)
        all_details.extend(issues)
        total_issues += len(issues)

        # Log issues
        if summary["status"] and summary["status"].startswith("error"):
            log("ERROR", f"Status  : {summary['status']}")
        elif len(issues) == 0:
            log("OK",    f"No issues found ✨")
        else:
            log("ISSUE", f"{len(issues)} issue(s) ditemukan:")
            for iss in issues:
                log("ISSUE", f"  [{iss['issue_type']}] {iss['detail']}")

        # Simpan summary real-time (tanpa duplicate_h1 dulu, diupdate nanti)
        append_summary(summary)
        append_details(issues)
        log("SAVE",  f"Baris disimpan ke CSV")

        if i < total:
            log("DELAY", f"Menunggu {DELAY_SECONDS} detik...")
            time.sleep(DELAY_SECONDS)

    # ── Post-process: Duplicate H1 ────────────────────────────
    print("\n" + "═" * 60)
    log("INFO", "Mendeteksi duplicate H1 lintas halaman...")
    duplicate_detail_rows = flag_duplicate_h1(all_summaries)

    # Rewrite summary CSV dengan kolom has_duplicate_h1 yang sudah benar
    pd.DataFrame(all_summaries)[SUMMARY_COLS].to_csv(
        OUTPUT_SUMMARY, index=False, encoding="utf-8"
    )
    # Append duplicate H1 rows ke detail CSV
    append_details(duplicate_detail_rows)

    dup_count = len(duplicate_detail_rows)
    if dup_count:
        log("ISSUE", f"Duplicate H1 ditemukan di {dup_count} halaman")
    else:
        log("OK",    "Tidak ada duplicate H1 lintas halaman")

    # ── Final Summary ─────────────────────────────────────────
    print("\n" + "═" * 60)
    clean = sum(1 for s in all_summaries if s["issue_count"] == 0 and not s.get("has_duplicate_h1"))
    log("DONE",  f"Selesai! {total} URL diaudit | 🚨 Total issues: {total_issues + dup_count} | ✅ Clean: {clean}")
    log("SAVE",  f"Summary  → {OUTPUT_SUMMARY}")
    log("SAVE",  f"Detail   → {OUTPUT_DETAIL}")

main()

## 👀 Preview Hasil

In [ ]:
import pandas as pd
import os

if not os.path.exists(OUTPUT_SUMMARY):
    print("❌ File output belum tersedia. Jalankan cell audit terlebih dahulu.")
else:
    summary_df = pd.read_csv(OUTPUT_SUMMARY)
    detail_df  = pd.read_csv(OUTPUT_DETAIL)

    issue_cols = [
        "has_missing_title", "has_missing_meta_desc",
        "has_missing_h1", "has_multiple_h1", "has_duplicate_h1",
        "has_hierarchy_issue",
    ]

    print("═" * 50)
    print("📊 AUDIT SUMMARY")
    print("═" * 50)
    print(f"  Total URL diaudit   : {len(summary_df)}")
    print(f"  ✅ Halaman bersih    : {(summary_df['issue_count'] == 0).sum()}")
    print(f"  🚨 Halaman bermasalah: {(summary_df['issue_count'] > 0).sum()}")
    print()
    print("📋 BREAKDOWN ISSUES:")
    labels = {
        "has_missing_title":     "Missing title",
        "has_missing_meta_desc": "Missing meta description",
        "has_missing_h1":        "Missing H1",
        "has_multiple_h1":       "Multiple H1",
        "has_duplicate_h1":      "Duplicate H1 (lintas halaman)",
        "has_hierarchy_issue":   "Heading hierarchy violation",
    }
    for col, label in labels.items():
        count = summary_df[col].sum() if col in summary_df.columns else 0
        bar   = "🔴" * int(count) if count <= 10 else f"🔴 x{count}"
        print(f"  {label:<35}: {count:>3}  {bar}")

    print("\n" + "═" * 50)
    print("📄 SUMMARY TABLE (semua URL)")
    print("═" * 50)
    display(summary_df)

    print("\n" + "═" * 50)
    print("🚨 DETAIL ISSUES")
    print("═" * 50)
    if len(detail_df) == 0:
        print("  ✨ Tidak ada issue ditemukan!")
    else:
        display(detail_df)

## 💾 Download Hasil

In [ ]:
from google.colab import files
import os

for f in [OUTPUT_SUMMARY, OUTPUT_DETAIL]:
    if os.path.exists(f):
        print(f"📥 Mendownload {f}...")
        files.download(f)
    else:
        print(f"❌ {f} belum tersedia.")